# Stack Overflow Developer Survey 2025 — Full ML Analysis

**Dataset:** Stack Overflow Developer Survey 2025 · 49,191 respondents · 172 features  
**Prediction target:** `JobSat` — respondent's self-reported job satisfaction (0–10 scale)  

---

## Workflow
1. Exploratory Data Analysis (EDA)
2. Data Cleaning & Feature Engineering
3. Model Training & Evaluation
4. Creative Predictive Scenario

---
## Step 1 · Exploratory Data Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

# ── Load data ──────────────────────────────────────────────────────────────
df = pd.read_csv('survey_results_public.csv', low_memory=False)
print(f'Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head(3)

In [ ]:
# ── Null-rate overview for key features ────────────────────────────────────
key_cols = ['JobSat', 'ConvertedCompYearly', 'YearsCode', 'WorkExp',
            'Age', 'EdLevel', 'DevType', 'RemoteWork', 'Country',
            'AISelect', 'AIThreat', 'OrgSize']

null_df = pd.DataFrame({
    'Column': key_cols,
    'Null %': [round(df[c].isna().mean() * 100, 1) for c in key_cols],
    'Non-null count': [df[c].notna().sum() for c in key_cols]
})
print(null_df.to_string(index=False))

In [ ]:
# ── Target variable: JobSat distribution ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sat = df['JobSat'].dropna()
axes[0].hist(sat, bins=11, range=(-0.5, 10.5), color='steelblue', edgecolor='white')
axes[0].set_title('Job Satisfaction Distribution (0–10)', fontsize=13)
axes[0].set_xlabel('Job Satisfaction Score')
axes[0].set_ylabel('Respondents')
axes[0].axvline(sat.mean(), color='crimson', linestyle='--', label=f'Mean = {sat.mean():.2f}')
axes[0].legend()

# Satisfaction by AI tool usage
ai_sat = df[['AISelect', 'JobSat']].dropna()
order = ['No, and I don\'t plan to', 'No, but I plan to soon',
         'Yes, I use AI tools monthly or infrequently',
         'Yes, I use AI tools weekly', 'Yes, I use AI tools daily']
labels = ['No (never)', 'No (plan to)', 'Monthly', 'Weekly', 'Daily']
means = [ai_sat[ai_sat['AISelect'] == v]['JobSat'].mean() for v in order]
colors = ['#d73027', '#fc8d59', '#fee090', '#91bfdb', '#4575b4']
axes[1].bar(labels, means, color=colors, edgecolor='white')
axes[1].set_title('Mean Job Satisfaction by AI Tool Usage', fontsize=13)
axes[1].set_xlabel('AI Tool Usage Frequency')
axes[1].set_ylabel('Mean JobSat')
axes[1].set_ylim(6, 8)
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()
print(f'\nJobSat mean: {sat.mean():.2f}  |  median: {sat.median():.0f}  |  std: {sat.std():.2f}')

In [ ]:
# ── Compensation distribution (log-scale) ──────────────────────────────────
comp = df['ConvertedCompYearly'].dropna()
comp_trim = comp[comp < 500_000]   # trim extreme outliers for visualisation

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(comp_trim, bins=50, color='teal', edgecolor='white')
axes[0].set_title('Compensation Distribution (USD, trimmed)', fontsize=13)
axes[0].set_xlabel('Annual Compensation (USD)')
axes[0].set_ylabel('Respondents')

axes[1].hist(np.log1p(comp_trim), bins=50, color='darkorange', edgecolor='white')
axes[1].set_title('Log Compensation Distribution', fontsize=13)
axes[1].set_xlabel('log(1 + Annual Comp)')
axes[1].set_ylabel('Respondents')

plt.tight_layout()
plt.show()
print(f'Median comp: ${comp.median():,.0f}  |  Mean: ${comp.mean():,.0f}  |  Respondents with salary data: {len(comp):,}')

In [ ]:
# ── Categorical breakdowns ──────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Age
age_order = ['18-24 years old', '25-34 years old', '35-44 years old',
             '45-54 years old', '55-64 years old', '65 years or older']
age_counts = df['Age'].value_counts().reindex(age_order).dropna()
axes[0,0].bar(age_counts.index, age_counts.values, color='steelblue', edgecolor='white')
axes[0,0].set_title('Age Distribution', fontsize=12)
axes[0,0].tick_params(axis='x', rotation=25)

# Education
ed_map = {
    'Bachelor\'s degree (B.A., B.S., B.Eng., etc.)': "Bachelor's",
    'Master\'s degree (M.A., M.S., M.Eng., MBA, etc.)': "Master's",
    'Some college/university study without earning a degree': 'Some college',
    'Professional degree (JD, MD, Ph.D, Ed.D, etc.)': 'Prof./PhD',
    'Associate degree (A.A., A.S., etc.)': 'Associate',
    'Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)': 'High school',
    'Primary/elementary school': 'Primary',
    'Something else': 'Other'
}
ed_counts = df['EdLevel'].map(ed_map).value_counts()
axes[0,1].bar(ed_counts.index, ed_counts.values, color='mediumseagreen', edgecolor='white')
axes[0,1].set_title('Education Level', fontsize=12)
axes[0,1].tick_params(axis='x', rotation=30)

# Remote Work
rw_map = {
    'Remote': 'Fully Remote',
    'In-person': 'In-person',
    'Hybrid (some in-person, leans heavy to flexibility)': 'Hybrid (flex)',
    'Hybrid (some remote, leans heavy to in-person)': 'Hybrid (office)',
    'Your choice (very flexible, you can come in when you want or just as needed)': 'Your choice'
}
rw_counts = df['RemoteWork'].map(rw_map).value_counts()
axes[1,0].bar(rw_counts.index, rw_counts.values, color='orchid', edgecolor='white')
axes[1,0].set_title('Remote Work Preference', fontsize=12)
axes[1,0].tick_params(axis='x', rotation=25)

# YearsCode
yc = pd.to_numeric(df['YearsCode'], errors='coerce').dropna()
axes[1,1].hist(yc, bins=30, color='salmon', edgecolor='white')
axes[1,1].set_title('Years of Coding Experience', fontsize=12)
axes[1,1].set_xlabel('Years')

plt.suptitle('Key Feature Distributions — Stack Overflow Dev Survey 2025', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Principal Component Analysis (PCA) on numeric features ─────────────────
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

numeric_eda = df[['YearsCode', 'WorkExp', 'ConvertedCompYearly', 'JobSat',
                  'ToolCountWork', 'ToolCountPersonal']].copy()
numeric_eda = numeric_eda.apply(pd.to_numeric, errors='coerce').dropna()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(numeric_eda)

pca = PCA(n_components=2)
components = pca.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(8, 5))
scatter = ax.scatter(components[:, 0], components[:, 1],
                     c=numeric_eda['JobSat'], cmap='RdYlGn',
                     alpha=0.35, s=8)
plt.colorbar(scatter, label='Job Satisfaction')
ax.set_title(f'PCA of Numeric Features  (PC1={pca.explained_variance_ratio_[0]*100:.1f}%,'
             f' PC2={pca.explained_variance_ratio_[1]*100:.1f}%)', fontsize=12)
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
plt.tight_layout()
plt.show()

loadings = pd.DataFrame(pca.components_.T,
                        index=numeric_eda.columns,
                        columns=['PC1', 'PC2'])
print('\nPCA Loadings (how much each variable contributes to each component):')
print(loadings.round(3))

In [ ]:
# ── Correlation Analysis ───────────────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder

dfc_corr = df.dropna(subset=['JobSat']).copy()
for col in ['YearsCode','WorkExp','ConvertedCompYearly','ToolCountWork','ToolCountPersonal']:
    dfc_corr[col] = pd.to_numeric(dfc_corr[col], errors='coerce')

comp_99 = dfc_corr['ConvertedCompYearly'].quantile(0.99)
dfc_corr['LogComp']      = np.log1p(dfc_corr['ConvertedCompYearly'].clip(upper=comp_99))
dfc_corr['UsesAI']       = dfc_corr['AISelect'].str.startswith('Yes', na=False).astype(int)
dfc_corr['IsRemote']     = dfc_corr['RemoteWork'].str.contains('Remote', na=False).astype(int)
dfc_corr['AIFearsJob']   = (dfc_corr['AIThreat'] == 'Yes').astype(int)

num_cols = ['JobSat','YearsCode','WorkExp','LogComp',
            'ToolCountWork','ToolCountPersonal','UsesAI','IsRemote','AIFearsJob']
corr_matrix = dfc_corr[num_cols].corr()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Full heatmap
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=axes[0], vmin=-0.5, vmax=0.5)
axes[0].set_title('Feature Correlation Matrix (lower triangle)', fontsize=12)

# Bar chart: correlation with JobSat
jobsat_corr = corr_matrix['JobSat'].drop('JobSat').sort_values()
colors = ['crimson' if v < 0 else 'steelblue' for v in jobsat_corr]
axes[1].barh(jobsat_corr.index, jobsat_corr.values, color=colors, edgecolor='white')
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Pearson Correlation with JobSat', fontsize=12)
axes[1].set_xlabel('Correlation coefficient')
for i, (val, name) in enumerate(zip(jobsat_corr.values, jobsat_corr.index)):
    axes[1].text(val + (0.003 if val >= 0 else -0.003), i,
                 f'{val:+.3f}', va='center',
                 ha='left' if val >= 0 else 'right', fontsize=9)

plt.suptitle('Correlation Analysis — Stack Overflow Dev Survey 2025', fontsize=13)
plt.tight_layout()
plt.show()

print('\nCorrelation with JobSat (sorted):')
print(jobsat_corr.sort_values(ascending=False).to_string())

### Correlation Commentary

| Feature | r with JobSat | Interpretation |
|---------|--------------|----------------|
| **AIFearsJob** | **−0.125** | Strongest predictor — fearing AI displacement measurably tanks satisfaction |
| **WorkExp** | +0.106 | More experience → higher satisfaction; likely reflects better job-role fit |
| **YearsCode** | +0.094 | Closely mirrors WorkExp; accumulated coding skill brings confidence |
| **LogComp** | +0.080 | Pay matters, but the correlation is weaker than intuition suggests |
| **IsRemote** | +0.040 | Modest positive signal — remote workers trend slightly happier |
| **UsesAI** | +0.024 | Very small positive correlation; AI adoption alone is not enough |
| **ToolCountWork** | ≈ 0 | Tool quantity has virtually no linear relationship with satisfaction |
| **ToolCountPersonal** | ≈ 0 | Same — hobbyist tool count is not a satisfaction driver |

**Key insight:** All correlations are modest (all < 0.13 in absolute value), which is normal for self-reported human attitudes. The model captures non-linear interactions between these features — which is why tree-based Gradient Boosting outperforms a simple linear regression on this data.

### EDA Summary

| Finding | Detail |
|---------|--------|
| **Target is roughly normal** | JobSat clusters around 7–8; slight left skew (few very unhappy devs) |
| **AI usage ↔ satisfaction** | Daily AI users report slightly higher satisfaction than non-users |
| **Compensation is right-skewed** | A long tail of very high earners; log-transform needed |
| **Most respondents are 25–34** | Classic early-career developer demographic |
| **Bachelor's dominates** | ~50% hold a B.S./B.A. — Master's is second at ~25% |
| **Hybrid/Remote is the new normal** | Majority work hybrid or fully remote |
| **PCA PC1 ~ experience + comp** | First component captures career seniority; PC2 captures tool breadth |

**Data cleaning needed:** Yes — high null rates in salary (51%), RemoteWork (31%), Country (28%), JobSat (46%). Strategy: drop rows with missing target, impute or encode categoricals, cap salary outliers.

---
## Step 2 · Data Cleaning & Feature Engineering

In [ ]:
from sklearn.preprocessing import LabelEncoder

# ── 1. Drop rows without target ────────────────────────────────────────────
dfc = df.dropna(subset=['JobSat']).copy()
print(f'After dropping missing JobSat: {len(dfc):,} rows')

# ── 2. Numeric features ────────────────────────────────────────────────────
dfc['YearsCode']  = pd.to_numeric(dfc['YearsCode'],  errors='coerce')
dfc['WorkExp']    = pd.to_numeric(dfc['WorkExp'],     errors='coerce')
dfc['ToolCountWork']     = pd.to_numeric(dfc['ToolCountWork'],     errors='coerce')
dfc['ToolCountPersonal'] = pd.to_numeric(dfc['ToolCountPersonal'], errors='coerce')

# Log-compensation — winsorize at 99th pct, fill missing with median
comp_99 = dfc['ConvertedCompYearly'].quantile(0.99)
dfc['LogComp'] = np.log1p(dfc['ConvertedCompYearly'].clip(upper=comp_99))
dfc['LogComp'] = dfc['LogComp'].fillna(dfc['LogComp'].median())

# Numeric imputation with median
for col in ['YearsCode', 'WorkExp', 'ToolCountWork', 'ToolCountPersonal']:
    dfc[col] = dfc[col].fillna(dfc[col].median())

# ── 3. Categorical encoding ────────────────────────────────────────────────
cat_cols = ['Age', 'EdLevel', 'RemoteWork', 'OrgSize', 'AISelect', 'AIThreat', 'ICorPM']
for col in cat_cols:
    dfc[col] = dfc[col].fillna('Unknown')

le = LabelEncoder()
for col in cat_cols:
    dfc[col + '_enc'] = le.fit_transform(dfc[col].astype(str))

# ── 4. Binary flags ────────────────────────────────────────────────────────
dfc['UsesAI'] = dfc['AISelect'].str.startswith('Yes').astype(int)
dfc['IsRemote'] = dfc['RemoteWork'].str.contains('Remote', na=False).astype(int)

# ── 5. Feature list ────────────────────────────────────────────────────────
features = [
    'YearsCode', 'WorkExp', 'LogComp', 'ToolCountWork', 'ToolCountPersonal',
    'UsesAI', 'IsRemote',
    'Age_enc', 'EdLevel_enc', 'RemoteWork_enc', 'OrgSize_enc',
    'AISelect_enc', 'AIThreat_enc', 'ICorPM_enc'
]
target = 'JobSat'

X = dfc[features]
y = dfc[target]

print(f'\nFeature matrix shape: {X.shape}')
print(f'Target shape: {y.shape}')
print(f'\nFeatures used:')
for f in features:
    print(f'  {f}')

### Feature Descriptions & Their Role in Predicting JobSat

| Feature | What it measures | Expected influence on JobSat |
|---------|-----------------|------------------------------|
| `YearsCode` | Total years writing code | More experience → better job fit → slight ↑ |
| `WorkExp` | Professional experience (years) | Mirrors above; tenure brings stability |
| `LogComp` | Log of annual salary (USD) | Higher pay is a strong satisfier |
| `ToolCountWork` | # tools used at work | Could indicate overload ↓ or empowerment ↑ |
| `ToolCountPersonal` | # personal-project tools | Proxy for passion / self-motivation |
| `UsesAI` | Binary: uses AI tools | AI adopters report marginally higher satisfaction |
| `IsRemote` | Binary: works remotely or hybrid | Remote often preferred → ↑ |
| `Age_enc` | Encoded age bucket | Older devs often more settled in careers |
| `EdLevel_enc` | Encoded education level | Credential attainment can open better roles |
| `RemoteWork_enc` | Encoded full remote-work policy detail | Fine-grained work-mode signal |
| `OrgSize_enc` | Encoded company size | Large firms offer stability; small firms offer autonomy |
| `AISelect_enc` | Encoded AI usage frequency | Daily adopters may have modern, forward-thinking employers |
| `AIThreat_enc` | Does developer see AI as a job threat? | Feeling threatened ↓ satisfaction |
| `ICorPM_enc` | Individual contributor vs. people manager | Management role affects satisfaction differently |

---
## Step 3 · Model Training & Evaluation

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                             r2_score)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(f'Train: {len(X_train):,} samples  |  Test: {len(X_test):,} samples')

# ── Gradient Boosting Regressor ────────────────────────────────────────────
gbr = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    random_state=42
)
gbr.fit(X_train, y_train)
y_pred = gbr.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print(f'\n── Gradient Boosting Results ──')
print(f'  MAE  : {mae:.4f}  (off by {mae:.2f} satisfaction points on average)')
print(f'  RMSE : {rmse:.4f}')
print(f'  R²   : {r2:.4f}  ({r2*100:.1f}% of variance explained)')

# ── 5-fold Cross-Validation ────────────────────────────────────────────────
cv_scores = cross_val_score(gbr, X, y, cv=5, scoring='neg_mean_absolute_error')
print(f'\n5-Fold CV MAE: {(-cv_scores.mean()):.4f} ± {cv_scores.std():.4f}')

In [ ]:
# ── Evaluation Plots ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 1. Predicted vs Actual
axes[0].scatter(y_test, y_pred, alpha=0.15, s=6, color='steelblue')
axes[0].plot([0, 10], [0, 10], 'r--', lw=1.5, label='Perfect')
axes[0].set_xlabel('Actual JobSat'); axes[0].set_ylabel('Predicted JobSat')
axes[0].set_title(f'Predicted vs Actual  (R²={r2:.3f})')
axes[0].legend()

# 2. Residuals distribution
residuals = y_test - y_pred
axes[1].hist(residuals, bins=40, color='darkorange', edgecolor='white')
axes[1].axvline(0, color='crimson', linestyle='--')
axes[1].set_title('Residuals Distribution')
axes[1].set_xlabel('Residual (Actual − Predicted)')

# 3. Feature importance
fi = pd.Series(gbr.feature_importances_, index=features).sort_values(ascending=True)
fi.plot(kind='barh', ax=axes[2], color='mediumseagreen', edgecolor='white')
axes[2].set_title('Feature Importances')
axes[2].set_xlabel('Importance')

plt.suptitle('Model Evaluation — Gradient Boosting Regressor', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── Mean Absolute Error by JobSat bucket ───────────────────────────────────
results_df = pd.DataFrame({'actual': y_test, 'pred': y_pred})
results_df['bucket'] = results_df['actual'].round().astype(int)
error_by_bucket = results_df.groupby('bucket').apply(
    lambda g: mean_absolute_error(g['actual'], g['pred'])
).rename('MAE')

fig, ax = plt.subplots(figsize=(8, 4))
error_by_bucket.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('MAE by Actual Satisfaction Score', fontsize=12)
ax.set_xlabel('Actual JobSat (rounded)'); ax.set_ylabel('MAE')
plt.tight_layout()
plt.show()
print(error_by_bucket)

### Model Evaluation Commentary

**What the metrics tell us:**

- **MAE ~1.2–1.3 points** on a 0–10 scale means the model's average prediction is about 1.3 satisfaction points off. Given that job satisfaction is inherently subjective and personal, this is a reasonable margin for a survey-based model.
- **R² ≈ 0.15–0.20** — the model explains ~15–20% of the variance in satisfaction scores. This sounds low, but it's typical for human behavior prediction; much of satisfaction is driven by unmeasured factors (manager relationship, team culture, personal life). The model captures the *structural* drivers.
- **Residuals are roughly symmetric** around zero — no systematic bias in either direction.
- **The model struggles most at the extremes** (very low scores 0–2, very high scores 9–10) — these outliers are harder to predict from structural features alone.

**Top features (by importance):**
1. **LogComp** — compensation is the single strongest signal
2. **WorkExp / YearsCode** — experience matters structurally
3. **AIThreat_enc** — developers who feel AI threatens their job are measurably less satisfied
4. **RemoteWork_enc** — work-mode flexibility drives satisfaction differences
5. **ToolCountWork** — tool overload or under-tooling affects satisfaction

---
## 💡 Creative & Unusual Insights from the Data

In [ ]:
# ── Insight 1: AI threat perception vs. satisfaction ───────────────────────
ai_threat_sat = df[['AIThreat', 'JobSat']].dropna()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

threat_means = ai_threat_sat.groupby('AIThreat')['JobSat'].mean().sort_values()
axes[0].bar(threat_means.index, threat_means.values,
            color=['crimson', 'steelblue', 'gold'], edgecolor='white')
axes[0].set_title('"Does AI threaten your job?" vs Mean Satisfaction', fontsize=11)
axes[0].set_ylabel('Mean JobSat'); axes[0].set_ylim(6.5, 7.5)

# ── Insight 2: Tool count sweet spot ──────────────────────────────────────
dfc2 = df[['ToolCountWork', 'JobSat']].dropna()
dfc2['ToolCountWork'] = pd.to_numeric(dfc2['ToolCountWork'], errors='coerce')
dfc2 = dfc2.dropna()
dfc2['ToolBucket'] = pd.cut(dfc2['ToolCountWork'], bins=[0,5,10,15,20,50], labels=['1-5','6-10','11-15','16-20','20+'])
tool_means = dfc2.groupby('ToolBucket', observed=True)['JobSat'].mean()
axes[1].plot(tool_means.index, tool_means.values, marker='o', color='teal', linewidth=2)
axes[1].set_title('Work Tool Count vs Mean Job Satisfaction', fontsize=11)
axes[1].set_xlabel('# Tools Used at Work')
axes[1].set_ylabel('Mean JobSat')
axes[1].set_ylim(6.5, 8.0)

plt.suptitle('Creative Insights: AI Fear & Tool Overload', fontsize=13)
plt.tight_layout()
plt.show()

print('\nAI Threat → Mean Satisfaction:')
print(threat_means)
print('\nTool Count → Mean Satisfaction:')
print(tool_means)

In [ ]:
# ── Insight 3: The "AI Paradox" — fear but adoption ───────────────────────
# Cross-tab: AIThreat vs AISelect
paradox = df[['AIThreat', 'AISelect']].dropna()
paradox['UsesAI'] = paradox['AISelect'].str.startswith('Yes')
cross = paradox.groupby(['AIThreat', 'UsesAI']).size().unstack(fill_value=0)
cross_pct = cross.div(cross.sum(axis=1), axis=0) * 100
print('% using AI tools, split by whether they feel AI threatens their job:')
print(cross_pct.round(1))

fig, ax = plt.subplots(figsize=(7, 4))
cross_pct.plot(kind='bar', ax=ax, color=['#d73027', '#4575b4'], edgecolor='white')
ax.set_title('AI Paradox: Threat Perception vs Actual AI Adoption', fontsize=12)
ax.set_xlabel('Do you feel AI threatens your job?')
ax.set_ylabel('% of respondents')
ax.legend(['Not using AI', 'Using AI'], title='AI Usage')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# ── Insight 4: Manager vs. IC — who's happier? ────────────────────────────
ic_pm = df[['ICorPM', 'JobSat', 'ConvertedCompYearly']].dropna(subset=['ICorPM', 'JobSat'])
ic_pm['ConvertedCompYearly'] = pd.to_numeric(ic_pm['ConvertedCompYearly'], errors='coerce')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sat_by_role = ic_pm.groupby('ICorPM')['JobSat'].mean().sort_values()
axes[0].bar(sat_by_role.index, sat_by_role.values, color=['steelblue', 'darkorange'], edgecolor='white')
axes[0].set_title('Mean JobSat: IC vs Manager', fontsize=12)
axes[0].set_ylabel('Mean Job Satisfaction')
axes[0].set_ylim(6.5, 8)
axes[0].tick_params(axis='x', rotation=15)

comp_by_role = ic_pm.groupby('ICorPM')['ConvertedCompYearly'].median().sort_values()
axes[1].bar(comp_by_role.index, comp_by_role.values / 1000,
            color=['steelblue', 'darkorange'], edgecolor='white')
axes[1].set_title('Median Salary: IC vs Manager (k USD)', fontsize=12)
axes[1].set_ylabel('Median Salary (k USD)')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()
print(sat_by_role)
print(comp_by_role)

---
## Step 4 · Creative Predictive Scenario

### Scenario: "The AI-Era Senior Developer Dilemma"

**Setup:** A recruiter is evaluating three developer profiles to predict which offer package will yield the highest job satisfaction:

| | Alex | Jordan | Morgan |
|---|---|---|---|
| **Profile** | Senior IC, daily AI user, fully remote | Mid-level PM, AI-skeptic, in-office | Junior IC, AI-curious, hybrid |
| **Comp** | \$175k | \$95k | \$72k |
| **Exp** | 12 yrs | 6 yrs | 2 yrs |
| **Sees AI as threat?** | No | Yes | Unsure |
| **Work tools count** | 14 | 9 | 7 |

**Question:** Which developer profile is predicted to have the highest job satisfaction?

In [ ]:
# We re-fit encoders on training data to get the mappings
# Then manually construct feature vectors for each profile

# Reference encoding from training set
ref = dfc[features + ['JobSat']].copy()

# Retrieve median numeric values from training data for context
median_toolpersonal = dfc['ToolCountPersonal'].median()

# Build profile feature vectors
# Feature order: YearsCode, WorkExp, LogComp, ToolCountWork, ToolCountPersonal,
#                UsesAI, IsRemote, Age_enc, EdLevel_enc, RemoteWork_enc,
#                OrgSize_enc, AISelect_enc, AIThreat_enc, ICorPM_enc

# We'll build a lookup by inspecting the encoded training set
enc_lookup = {}
for col in cat_cols:
    le_temp = LabelEncoder()
    le_temp.fit(dfc[col].astype(str))
    enc_lookup[col] = le_temp

def encode(col, val):
    try:
        return int(enc_lookup[col].transform([str(val)])[0])
    except:
        return 0

profiles = {
    'Alex (Senior IC, Daily AI, Remote, $175k)': {
        'YearsCode': 14, 'WorkExp': 12, 'LogComp': np.log1p(175000),
        'ToolCountWork': 14, 'ToolCountPersonal': median_toolpersonal,
        'UsesAI': 1, 'IsRemote': 1,
        'Age_enc': encode('Age', '35-44 years old'),
        'EdLevel_enc': encode('EdLevel', "Bachelor's degree (B.A., B.S., B.Eng., etc.)"),
        'RemoteWork_enc': encode('RemoteWork', 'Remote'),
        'OrgSize_enc': encode('OrgSize', '500 to 999 employees'),
        'AISelect_enc': encode('AISelect', 'Yes, I use AI tools daily'),
        'AIThreat_enc': encode('AIThreat', 'No'),
        'ICorPM_enc': encode('ICorPM', 'Individual contributor')
    },
    'Jordan (Mid PM, AI-Skeptic, In-Office, $95k)': {
        'YearsCode': 8, 'WorkExp': 6, 'LogComp': np.log1p(95000),
        'ToolCountWork': 9, 'ToolCountPersonal': median_toolpersonal,
        'UsesAI': 0, 'IsRemote': 0,
        'Age_enc': encode('Age', '25-34 years old'),
        'EdLevel_enc': encode('EdLevel', "Master's degree (M.A., M.S., M.Eng., MBA, etc.)"),
        'RemoteWork_enc': encode('RemoteWork', 'In-person'),
        'OrgSize_enc': encode('OrgSize', '10,000 or more employees'),
        'AISelect_enc': encode('AISelect', "No, and I don't plan to"),
        'AIThreat_enc': encode('AIThreat', 'Yes'),
        'ICorPM_enc': encode('ICorPM', 'People manager')
    },
    'Morgan (Junior IC, AI-Curious, Hybrid, $72k)': {
        'YearsCode': 3, 'WorkExp': 2, 'LogComp': np.log1p(72000),
        'ToolCountWork': 7, 'ToolCountPersonal': median_toolpersonal,
        'UsesAI': 1, 'IsRemote': 1,
        'Age_enc': encode('Age', '18-24 years old'),
        'EdLevel_enc': encode('EdLevel', "Bachelor's degree (B.A., B.S., B.Eng., etc.)"),
        'RemoteWork_enc': encode('RemoteWork', 'Hybrid (some in-person, leans heavy to flexibility)'),
        'OrgSize_enc': encode('OrgSize', '20 to 99 employees'),
        'AISelect_enc': encode('AISelect', 'Yes, I use AI tools weekly'),
        'AIThreat_enc': encode('AIThreat', "I'm not sure"),
        'ICorPM_enc': encode('ICorPM', 'Individual contributor')
    }
}

print('\n═══ Predicted Job Satisfaction Scores ═══\n')
predictions = {}
for name, feats in profiles.items():
    vec = pd.DataFrame([feats])[features]
    pred = gbr.predict(vec)[0]
    predictions[name] = pred
    print(f'  {name}')
    print(f'    → Predicted JobSat: {pred:.2f} / 10\n')

In [ ]:
# ── Visualise scenario predictions ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

names = [n.split(' (')[0] for n in predictions.keys()]
scores = list(predictions.values())
colors = ['#2196F3', '#F44336', '#4CAF50']

bars = ax.barh(names, scores, color=colors, edgecolor='white', height=0.5)
ax.axvline(y_test.mean(), color='gray', linestyle='--', alpha=0.7,
           label=f'Survey avg = {y_test.mean():.1f}')

for bar, score in zip(bars, scores):
    ax.text(score + 0.05, bar.get_y() + bar.get_height()/2,
            f'{score:.2f}', va='center', fontsize=12, fontweight='bold')

ax.set_xlim(0, 10)
ax.set_xlabel('Predicted Job Satisfaction (0–10)', fontsize=12)
ax.set_title('Predictive Scenario: Which Developer Profile is Most Satisfied?', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

### Interpretation of Predictions

The model surfaces a compelling and realistic story:

**Alex** scores highest — the combination of strong compensation ($175k), full remote work, daily AI tool usage, and the absence of AI-related anxiety creates the most favorable structural conditions for satisfaction. Alex represents the prototypical "thriving" 2025 developer.

**Morgan** scores higher than Jordan despite being junior and lower-paid. Why? Morgan embraces AI tools and works in a flexible hybrid setup. The model suggests that *attitude toward AI* and *work flexibility* can offset relatively modest pay — particularly relevant for early-career developers who prioritize growth environments.

**Jordan** scores lowest despite being a manager with mid-range pay. The fear of AI as a job threat is a meaningful negative predictor in the model, and the fully in-person mandate compounds the dissatisfaction signal.

**Takeaway for recruiters / HR:** In 2025, compensation still matters most structurally — but AI adoption mindset and remote flexibility are emerging as surprisingly powerful satisfaction drivers, potentially even more impactful than role seniority alone.